# Run Controls 1-3 across all four models (Google Colab)

Upload `all_models_colab.zip` when prompted below. Each model section then shows you **the actual code that runs** (syntax-highlighted, read it before running), runs it as a real subprocess via `!python` (this sidesteps `ProcessPoolExecutor`/`__file__` quirks that come from running multiprocessing code pasted directly into a notebook cell), then displays **that model's own figure** inline -- not just the combined one at the end.

Run cells **in order, one at a time** (not all queued up together) -- each script already parallelizes across every CPU core Colab gives you internally, so running several at once just makes them compete instead of finishing faster.

All four default to `n_seeds=20`, so each run cell takes a while -- you'll see per-seed progress lines as it goes.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick all_models_colab.zip

In [ ]:
!unzip -o all_models_colab.zip -d /content/
%cd /content
!pip install -q numpy matplotlib scipy

from IPython.display import Code, Image, display

def show_code(path):
    display(Code(filename=path, language='python'))

def show_figure(path):
    display(Image(path))

## 1. Updated model v1

One flat, self-contained script -- the simulation, the three controls, the stats, and the plotting all in one file.

In [ ]:
show_code("attractor version1/attractor_control_analysis.py")

In [ ]:
!python "attractor version1/attractor_control_analysis.py"

In [ ]:
show_figure("attractor version1/output/full_control_comparison.png")

## 2. Cued attractor

This one's a proper package: `model.py` (the network equations), `task.py` (the vocabulary), `experiment.py` (turns trials into blocks into a full session, and is where the three controls are implemented), `analysis.py` (behavioural summaries), and `run_controls.py` below, which drives all of it. Showing `run_controls.py` since it's the file that actually decides what gets run and how the results get read -- open the others in Colab's file browser (folder icon, left sidebar) if you want to see the network equations themselves.

In [ ]:
show_code("cued_attractor/run_controls.py")

In [ ]:
!python cued_attractor/run_controls.py

In [ ]:
show_figure("cued_attractor/output/controls/controls_comparison.png")

## 3. Gated attractor

Same shape as the cued attractor, plus a gating population that learns to suppress the task-irrelevant feature. Uses the `2cpr_slowW3` parameter preset -- the default gating preset (`2cpr_gating_units`) leaves this model at chance accuracy due to a documented routing-instability issue (see `gated_attractor/model_outline.md`); `slowW3` is that package's own validated fix.

In [ ]:
show_code("gated_attractor/run_controls.py")

In [ ]:
!python gated_attractor/run_controls.py

In [ ]:
show_figure("gated_attractor/output/controls/controls_comparison.png")

## 4. Original (published) model

No cue mechanism -- each block re-teaches its rule from scratch. Only runs baseline plus the adapted Control 3 (withhold one stimulus combination until partway through, then let it appear); Controls 1/2 have no equivalent here, see the script's own docstring for why.

In [ ]:
show_code("original model/run_baseline_and_control3.py")

In [ ]:
!python "original model/run_baseline_and_control3.py"

In [ ]:
show_figure("original model/output/controls_comparison.png")

## 5. Combine all four into one comparison

Fast -- just reads the four `controls_summary.json` files written above, doesn't run any simulations itself.

In [ ]:
show_code("compare_all_models.py")

In [ ]:
!python compare_all_models.py

In [ ]:
show_figure("output/all_models_comparison.png")

## 6. Congruency-matched Control 3 (the proper novelty test)

The Control 3 numbers in sections 1-5 above have a confound: every model's omitted combo was (cue A1, green-square) -- the *congruent* stimulus, which is inherently easier regardless of whether it was trained. This section reruns Control 3 with an *incongruent* omitted combo instead (cue A1, green-circle), and -- critically -- compares it only against **other trained incongruent combos**, not the full mixed pool. That isolates "was this specific pairing ever trained" from "congruent trials are easier in general," which the original version conflated.

No figure here -- this script only prints stats and writes a JSON summary, so the run cell's own output below is the complete result.

In [ ]:
show_code("congruency_matched_control3.py")

In [ ]:
!python congruency_matched_control3.py

## 7. (Optional) Download all results back to your machine

Colab's filesystem is wiped when the session ends -- run this if you want to keep the figures/JSON.

In [ ]:
!zip -r all_models_results.zip output \
  "attractor version1/output" \
  cued_attractor/output \
  gated_attractor/output \
  "original model/output"
from google.colab import files
files.download('all_models_results.zip')